> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 10 · Notebook 10 — Deep-learning building blocks: causality, anomalies, uncertainty and the honest comparison

**Sessions:** S19–S20 (Deep-learning strategies & advanced analysis · Optimization & the honest comparison) · [Lesson plan](../../docs/lessons/PART_10_MACHINE_LEARNING.md) · graded labs in [`labs/part10/`](../../labs/part10/)

**You will:**
1. Write a causal convolution and prove it cannot see the future.
2. Score market days with an autoencoder trained on calm days.
3. Estimate uncertainty with MC dropout and forecast quantiles with the pinball loss.
4. Report seed variance, not the best seed.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with a known structure (a hidden regime, a known autocorrelation, planted importances, pure noise), so you can see what a model can learn, and that it learns nothing from noise.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p10lib.py is in notebooks/part10/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p10lib as p

p.use_course_style()
import torch
import p10dl as dl

## 1. Causal convolutions

A convolution over time with symmetric padding mixes `t−1`, `t` and `t+1`: the output at `t` sees the future. A **causal** convolution pads only on the **left**, by `(kernel − 1)·dilation` zeros, so the output at `t` depends on inputs up to `t` and the length is unchanged. `nn.functional.pad(x, (left, right))` pads the last dimension.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
from torch import nn

class MyCausalConv1d(nn.Module):
    def __init__(self, c_in, c_out, kernel=3, dilation=1):
        super().__init__()
        self.pad = (kernel - 1) * dilation
        self.conv = nn.Conv1d(c_in, c_out, kernel, dilation=dilation)

    def forward(self, x):                                            # x: (batch, channels, time)
        return self.conv(nn.functional.pad(x, (self.pad, 0)))

dl.set_seed(0)
x_in = torch.randn(2, 3, 30)
outs = []
for kernel, dilation in ((3, 1), (3, 4), (2, 8)):
    mine_conv, ref_conv = MyCausalConv1d(3, 4, kernel, dilation), dl.CausalConv1d(3, 4, kernel, dilation)
    ref_conv.load_state_dict(mine_conv.state_dict())                  # the same weights
    with torch.no_grad():
        got = p.attempt(lambda m=mine_conv: m(x_in).numpy())
        outs.append((got, ref_conv(x_in).numpy()))
p.check("CausalConv1d", [o[0] for o in outs], [o[1] for o in outs])

future = x_in.clone()
future[:, :, 20:] += 5.0                                 # change only the inputs from t = 20 on
with torch.no_grad():
    causal = (ref_conv(future) - ref_conv(x_in)).abs()[:, :, :20].max().item()
    centered_conv = nn.Conv1d(3, 4, 3, padding=1)
    centered = (centered_conv(future) - centered_conv(x_in)).abs()[:, :, 19].max().item()
print(f"outputs before t = 20 change by {causal:.1e} with causal padding, and by {centered:.2f} at t = 19 with centered padding")

Exactly zero: the causal output cannot react to anything after `t`. Put this test in your test suite for every sequence model; a centered convolution fails it immediately.

## 2. An autoencoder as an anomaly detector

Train a small autoencoder (6 → 16 → 3 → 16 → 6) to reconstruct **calm** days' features from the regime market. On days unlike its training data it reconstructs poorly: the mean squared reconstruction error of the standardized row (standardized with the **training** mean and std) is the anomaly score.

In [ ]:
mk = p.regime_market()
r = np.log(mk["close"]).diff()
F = pd.DataFrame({"vol5": r.rolling(5).std(), "vol20": r.rolling(20).std(), "ret5": r.rolling(5).sum(),
                  "ret20": r.rolling(20).sum(), "absr": r.abs(), "er20": r.rolling(20).sum().abs() / r.abs().rolling(20).sum()}).dropna()
regime = mk["regime"].loc[F.index].map({0: "chop", 1: "trend"})
calm_days = F[regime == "trend"].iloc[:600]
ae, mu, sd = dl.fit_autoencoder(calm_days.to_numpy())

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def anomaly_scores(model, X, mu, sd):
    Z = torch.tensor((X - mu) / sd, dtype=torch.float32)
    model.eval()
    with torch.no_grad():
        return ((model(Z) - Z) ** 2).mean(1).numpy()

mine = p.check("anomaly_scores", p.attempt(anomaly_scores, ae, F.to_numpy(), mu, sd), dl.anomaly_scores(ae, F.to_numpy(), mu, sd))
score = pd.Series(mine, index=F.index)
from sklearn.metrics import roc_auc_score
print("median anomaly score by hidden regime:", score.groupby(regime).median().round(3).to_dict(),
      f"| AUC for spotting chop days: {roc_auc_score((regime == 'chop').astype(int), score):.2f}")

Trained on calm days only, the autoencoder flags choppy days without ever seeing a label. In the platform this is a monitoring signal ("today doesn't look like the data the models were trained on"), not a trading signal.

## 3. Uncertainty and quantiles

**MC dropout** keeps dropout on at prediction time and runs many passes: the spread of the outputs is a rough uncertainty. It grows on inputs unlike the training data:

In [ ]:
x = p.ar1(3000, 0.6)
L = 20
ds = dl.WindowDataset(x[:-1], x[1:], L, normalize=False)
splits = dl.time_splits(len(ds), L)
tr, va, te = dl.loaders(ds, splits)
dl.set_seed(0)
mlp = dl.MLP(1, L, 32, dropout=0.2)
dl.train(mlp, tr, va, epochs=30, patience=4)
windows = torch.stack([ds[i][0] for i in splits[2]])
_, std_normal = dl.mc_dropout(mlp, windows, 50)
_, std_extreme = dl.mc_dropout(mlp, windows[:50] * 4, 50)
print(f"MC-dropout std: {std_normal.mean():.2f} on test windows, {std_extreme.mean():.2f} on windows 4× larger than anything seen")

To forecast a **quantile** instead of the mean, train with the pinball loss: with error `e = y − pred`, the loss is the mean of `max(q·e, (q − 1)·e)` (`torch.maximum`). It is minimized when a share `q` of the targets lies below the prediction.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def pinball_loss(pred, y, q):
    e = y - pred
    return torch.mean(torch.maximum(q * e, (q - 1) * e))

pred, target = torch.tensor([0.0, 1.0, 2.0, 3.0]), torch.tensor([1.0, 1.0, 0.0, 5.0])
mine = [p.attempt(lambda q=q: pinball_loss(pred, target, q).item()) for q in (0.1, 0.5, 0.9)]
mine = p.check("pinball_loss", mine, [dl.pinball_loss(pred, target, q).item() for q in (0.1, 0.5, 0.9)])
print("pinball losses for q = 0.1, 0.5, 0.9:", np.round(mine, 3))

In [ ]:
rows = {}
for q in (0.1, 0.5, 0.9):
    dl.set_seed(0)
    model = dl.MLP(1, L, 16)
    dl.train(model, tr, va, epochs=30, patience=4, loss_fn=lambda a, b, q=q: dl.pinball_loss(a, b, q))
    pred, y_test = dl.predict(model, te)
    rows[f"q = {q}"] = np.mean(y_test <= pred)
pd.Series(rows, name="share of test targets below the forecast").round(3).to_frame()

## 4. The honest comparison

A deep model's result depends on its seed. Report the spread over seeds, next to the baseline, on the same test windows:

In [ ]:
cors = []
for seed in range(3):
    dl.set_seed(seed)
    lstm = dl.LSTMRegressor(1, 16)
    dl.train(lstm, tr, va, epochs=30, patience=4)
    pred, y_test = dl.predict(lstm, te)
    cors.append(np.corrcoef(pred, y_test)[0, 1])
naive = np.corrcoef(x[:-1][splits[2] + L - 1], y_test)[0, 1]
print(f"LSTM over 3 seeds: {np.round(cors, 3)} (mean {np.mean(cors):.3f});  naive rule: {naive:.3f}")

Three seeds, three slightly different results, all at or just below the one-line rule. The honest report says exactly that; the dishonest one shows the best seed on a different test window. Tuning (learning rate, window, hidden size) multiplies the trials: count them all.

## Wrap-up

* Test causality directly: perturb the future, the past outputs must not move.
* Autoencoders make good "this looks unusual" monitors.
* MC dropout and quantile losses give uncertainty; conformal intervals give coverage.
* Seeds, baselines and trial counts belong in every deep-learning result.
* Graded version: `labs/part10/week37_deep` (`CausalConv1d`, `TCNRegressor`, `pinball_loss`, `mc_dropout`, `AutoEncoder`, `anomaly_scores`).